# IMPORTS

In [133]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import to_hex
from tqdm import tqdm, trange

import time
from IPython.display import display
import os
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE

import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from collections import Counter
from sklearn.metrics import make_scorer, precision_score, recall_score, accuracy_score, precision_recall_curve, roc_auc_score, confusion_matrix
from sklearn.metrics import average_precision_score, log_loss, f1_score, brier_score_loss, balanced_accuracy_score

from threadpoolctl import threadpool_limits


import torch
import torch.nn as nn

import lime
import lime.lime_tabular
import json, math, pathlib


from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LassoCV, Lasso
import warnings
from sklearn.pipeline import Pipeline



os.environ["OMP_NUM_THREADS"] = "1"  # quiet MKL/MKL-threads warning on Windows

# DATA: radiomics + tabular_data

## Methods: import data from csv's, clean and split

In [134]:
from get_data import get_data

# pipeline_core.py
from __future__ import annotations
import os, json, joblib, re, warnings
from dataclasses import dataclass
from typing import Any, Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import f_classif


### Dataframe-preserving helpers

In [135]:
# ========== DataFrame-preserving helpers ==========

class DFStandardScaler(StandardScaler):
    """StandardScaler that returns a DataFrame with original column names."""
    def __init__(self, with_mean=False, with_std=True, copy=True):
        super().__init__(with_mean=with_mean, with_std=with_std, copy=copy)
        self._cols = None
        self._index = None
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self._cols = list(X.columns)
        return super().fit(X, y)
    def transform(self, X):
        Xt = super().transform(X)
        if self._cols is not None:
            idx = getattr(X, "index", None)
            return pd.DataFrame(Xt, columns=self._cols, index=idx)
        return Xt

class VarianceThresholdNames(VarianceThreshold):
    """VarianceThreshold that preserves column names and returns a DataFrame."""
    def __init__(self, threshold=0.0):
        super().__init__(threshold=threshold)
        self._in_cols = None
        self.out_cols_ = None
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self._in_cols = list(X.columns)
        return super().fit(X, y)
    def transform(self, X):
        Xt = super().transform(X)
        if self._in_cols is not None:
            mask = self.get_support()
            self.out_cols_ = [c for c, m in zip(self._in_cols, mask) if m]
            return pd.DataFrame(Xt, columns=self.out_cols_, index=getattr(X, "index", None))
        return Xt

import numpy as np
import pandas as pd
from scipy import stats

def cv_mean_std_ci(df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Returns a table with mean, std, and (lower, upper) CI for each numeric column.
    CI is computed as: mean ± t_{k-1, 1-alpha/2} * std/sqrt(k)
    """
    df_num = df.select_dtypes(include=[np.number]).copy()
    k = df_num.shape[0]
    if k < 2:
        raise ValueError("Need at least 2 folds to compute CI.")

    means = df_num.mean()
    stds  = df_num.std(ddof=1)
    se    = stds / np.sqrt(k)
    tcrit = stats.t.ppf(1 - alpha/2, df=k-1)

    ci_low  = means - tcrit * se
    ci_high = means + tcrit * se

    out = pd.DataFrame({
        "mean": means,
        "std": stds,
        "ci_low": ci_low,
        "ci_high": ci_high,
    })

    # nice formatted column if you want to print directly
    out["mean_ci95"] = out.apply(lambda r: f"{r['mean']:.4f} [{r['ci_low']:.4f}, {r['ci_high']:.4f}]", axis=1)
    return out

### UniqueThenTopK

In [136]:
# ========== ANOVA unique-then-topK (with logs for contrasts) ==========

class UniqueThenTopK(BaseEstimator, TransformerMixin):
    """
    1) Score all features (ANOVA).
    2) Collapse contrasts FIRST: for each base_id (name without first token), keep best-contrast.
    3) Then top-k per group (group by 'class', 'contrast' or 'class_contrast').
    4) ALWAYS KEEP any columns listed in `always_keep` or matching `always_keep_regex`.
    Assumes names like {CONTRAST}_{...}. Class inferred from tokens in:
      {FirstOrder, Shape, GLCM, GLRLM, GLSZM, GLDM, NGTDM}. Non-matching → 'UNK'.
    """
    CLASS_SET = {"firstorder","shape","glcm","glrlm","glszm","gldm","ngtdm"}
    def __init__(self, k_per_group=20, groupby="class",
                 score_func=f_classif, max_total=None,
                 always_keep=None, always_keep_regex=None,
                 verbose=True, show_examples=10):
        self.k_per_group = int(k_per_group)
        self.groupby = groupby
        self.score_func = score_func
        self.max_total = max_total
        self.always_keep = set(always_keep) if always_keep is not None else set()
        self.always_keep_regex = re.compile(always_keep_regex) if isinstance(always_keep_regex, str) else None
        self.verbose = verbose
        self.show_examples = int(show_examples)

    # ---- helpers ----
    def _parse_meta(self, columns):
        cols = [str(c) for c in columns]
        rows = []
        for name in cols:
            if "_" in name:
                contrast, rest = name.split("_", 1)
            else:
                contrast, rest = "UNK", name
            tokens = rest.split("_")
            cls = "UNK"
            for t in tokens:
                if t.lower() in self.CLASS_SET:
                    cls = t  # keep original casing
                    break
            rows.append({"name": name, "contrast": contrast, "base_id": rest, "cls": cls})
        return pd.DataFrame(rows)

    def _safe_scores(self, X, y):
        var = np.nanstd(X, axis=0)
        nonconst = var > 0
        scores = np.zeros(X.shape[1], dtype=float)
        if nonconst.any():
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                out = self.score_func(X[:, nonconst], y)
            s = out[0] if isinstance(out, (tuple, list)) else out
            s = np.asarray(s, dtype=float)
            scores[nonconst] = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)
        return scores, nonconst.sum(), (~nonconst).sum()

    def _group_series(self, df):
        if self.groupby == "class":
            return df["cls"]
        elif self.groupby == "contrast":
            return df["contrast"]
        elif self.groupby == "class_contrast":
            return df["cls"] + "|" + df["contrast"]
        else:
            raise ValueError("groupby must be one of {'class','contrast','class_contrast'}")

    def _always_keep_mask(self, columns):
        cols = [str(c) for c in columns]
        mask = np.array([c in self.always_keep for c in cols], dtype=bool)
        if self.always_keep_regex is not None:
            mask = mask | np.array([bool(self.always_keep_regex.search(c)) for c in cols], dtype=bool)
        return mask

    # ---- sklearn API ----
    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame): X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
    
        # always-keep mask
        ak_mask = self._always_keep_mask(self.columns_)
        ak_idx = np.where(ak_mask)[0]
        ak_names = [self.columns_[i] for i in ak_idx]
    
        # working set (non always-keep)
        work_cols = [c for i, c in enumerate(self.columns_) if not ak_mask[i]]
        X_work = X[work_cols]
        meta = self._parse_meta(work_cols)
    
        # Step 1: ANOVA scores
        scores_all, n_nonconst, n_const = self._safe_scores(X_work.values, y)
        meta["score"] = scores_all
        if self.verbose:
            print(f"[topk] scored {len(work_cols)} features (non-const={n_nonconst}, const={n_const})")
    
        # Step 2: best-contrast per base_id
        keep_idx_unique = []
        dropped = 0
        for _, sub in meta.groupby("base_id"):
            j_local = int(sub["score"].values.argmax())
            chosen_idx = int(sub.index[j_local])
            keep_idx_unique.append(chosen_idx)
            dropped += (len(sub) - 1)
        uniq_mask = np.zeros(len(meta), dtype=bool)
        uniq_mask[np.asarray(keep_idx_unique, dtype=int)] = True
        if self.verbose:
            print(f"[topk] collapse contrasts: bases={meta['base_id'].nunique()} | kept={uniq_mask.sum()} | dropped={dropped}")
    
        # Pre-selection counts (after uniqueness)
        meta_u = meta.loc[uniq_mask].copy()
        meta_u["group"] = self._group_series(meta_u)
        pre_counts = meta_u["group"].value_counts().sort_index()
    
        # Step 3: top-k per group
        keep_idx_topk = []
        for _, sub in meta_u.groupby("group"):
            k = min(self.k_per_group, len(sub))
            keep_idx_topk.extend(sub["score"].nlargest(k).index.tolist())
    
        # ---- requested compact per-subset logging ----
        if self.verbose:
            post_counts = (
                meta_u.loc[meta_u.index.isin(keep_idx_topk), "group"]
                      .value_counts()
                      .reindex(pre_counts.index, fill_value=0)
            )
            for g in pre_counts.index:
                print(f"{int(pre_counts[g])} -> {int(post_counts[g])}  ({g})")
    
        # Final mask over working columns
        final_work_mask = np.zeros(len(meta), dtype=bool)
        final_work_mask[np.asarray(keep_idx_topk, dtype=int)] = True
    
        # Optional global cap
        if self.max_total is not None and final_work_mask.sum() + len(ak_idx) > self.max_total:
            candidates = meta.loc[final_work_mask]
            keep_global = candidates["score"].nlargest(self.max_total - len(ak_idx)).index
            final_work_mask[:] = False
            final_work_mask[np.asarray(keep_global, dtype=int)] = True
    
        # Build final selection mask over ALL columns
        final_mask = np.zeros(len(self.columns_), dtype=bool)
        final_mask[ak_idx] = True  # always keep
        work_idx_map = {c: i for i, c in enumerate(work_cols)}
        for i, c in enumerate(work_cols):
            if final_work_mask[work_idx_map[c]]:
                final_mask[self.columns_.index(c)] = True
    
        # Save
        self.meta_ = meta
        self.selected_mask_ = final_mask
        self.selected_features_ = list(pd.Series(self.columns_)[final_mask])
    
        if self.verbose and ak_names:
            print(f"[topk] always-kept (not grouped): {sorted([c for c in ak_names if c in self.columns_])}")
    
        return self



    def transform(self, X):
        if not isinstance(X, pd.DataFrame): X = pd.DataFrame(X)
        return X.loc[:, self.selected_features_]

    def get_support(self, indices=False):
        if indices:
            return np.where(self.selected_mask_)[0]
        return self.selected_mask_

### GroupedCoeThrLassoSelector

In [137]:
class GroupedCoeThrLassoSelector(BaseEstimator, TransformerMixin):
    """
    Apply Coe-Thr-Lasso feature selection *within each group of features*.
    Groups are defined by `groupby`: {"class", "contrast", "class_contrast"}.
    """

    CLASS_SET = {"firstorder","shape","glcm","glrlm","glszm","gldm","ngtdm"}

    def __init__(self, alpha_grid=None, random_state=42, test_size=0.5,
                 max_iter=5000, groupby="class", k_per_group=None):
        self.alpha_grid = alpha_grid if alpha_grid is not None else np.logspace(-3, 0, 50)
        self.random_state = random_state
        self.test_size = test_size
        self.max_iter = max_iter
        self.groupby = groupby
        self.k_per_group = k_per_group  # optional cap per group

    # ---------- helpers ----------
    def _parse_meta(self, columns):
        cols = [str(c) for c in columns]
        rows = []
        for name in cols:
            if "_" in name:
                contrast, rest = name.split("_", 1)
            else:
                contrast, rest = "UNK", name
            tokens = rest.split("_")
            cls = "UNK"
            for t in tokens:
                if t.lower() in self.CLASS_SET:
                    cls = t
                    break
            rows.append({
                "name": name,
                "contrast": contrast,
                "base_id": rest,
                "cls": cls,
            })
        meta = pd.DataFrame(rows)
        if self.groupby == "class":
            meta["group"] = meta["cls"]
        elif self.groupby == "contrast":
            meta["group"] = meta["contrast"]
        elif self.groupby == "class_contrast":
            meta["group"] = meta["cls"] + "|" + meta["contrast"]
        else:
            meta["group"] = "all"
        return meta

    def _coe_thr_lasso(self, Xg, y):
        """Run Coe-Thr-Lasso on one group."""
        # Variance filter
        var_mask = Xg.var(axis=0) > 0
        if not var_mask.any():
            return np.array([False]*Xg.shape[1]), np.zeros(Xg.shape[1])

        X1 = Xg.loc[:, var_mask]
        # Welch’s t-test
        F, p = f_classif(X1.values, y)
        sig_mask = p < 0.05
        if not sig_mask.any():
            return np.array([False]*Xg.shape[1]), np.zeros(Xg.shape[1])
        X2 = X1.loc[:, sig_mask]

        # Lasso
        lcv = LassoCV(alphas=self.alpha_grid, cv=5,
                      random_state=self.random_state,
                      max_iter=self.max_iter).fit(X2.values, y)
        alpha_opt = lcv.alpha_

        lasso = Lasso(alpha=alpha_opt, random_state=self.random_state,
                      max_iter=self.max_iter).fit(X2.values, y)
        coefs = lasso.coef_
        coef_mask = coefs != 0
        if not coef_mask.any():
            return np.array([False]*Xg.shape[1]), np.zeros(Xg.shape[1])

        X3 = X2.loc[:, coef_mask]
        coefs3 = coefs[coef_mask]

        # Threshold search
        hmax, hmin = np.max(np.abs(coefs3)), np.min(np.abs(coefs3))
        X_tr, X_val, y_tr, y_val = train_test_split(
            X3.values, y, test_size=self.test_size,
            random_state=self.random_state, stratify=y
        )

        best_thr, best_mse = None, np.inf
        for thr in np.arange(hmin, hmax, 0.001):
            mask = np.abs(coefs3) > thr
            if not mask.any():
                continue
            X_thr, X_val_thr = X_tr[:, mask], X_val[:, mask]
            m = Lasso(alpha=alpha_opt, random_state=self.random_state,
                      max_iter=self.max_iter).fit(X_thr, y_tr)
            mse = np.mean((m.predict(X_val_thr) - y_val) ** 2)
            if mse < best_mse:
                best_mse, best_thr = mse, thr

        final_mask_local = np.abs(coefs3) > (best_thr if best_thr is not None else hmin)
        selected_cols = set(X3.columns[final_mask_local])

        # Rebuild mask over original group
        sel_mask = np.array([c in selected_cols for c in Xg.columns])
        scores = np.zeros(Xg.shape[1])
        scores[sel_mask] = np.abs(coefs3[final_mask_local])
        return sel_mask, scores

    # ---------- sklearn API ----------
    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        meta = self._parse_meta(self.columns_)

        sel_mask = np.zeros(len(self.columns_), dtype=bool)
        scores_all = np.zeros(len(self.columns_), dtype=float)

        for g, idx in meta.groupby("group").groups.items():
            idx = list(idx)
            Xg = X.iloc[:, idx]
            sel_local, scores_local = self._coe_thr_lasso(Xg, y)
            sel_mask[idx] = sel_local
            scores_all[idx] = scores_local

            # optional: cap number per group
            if self.k_per_group is not None and sel_local.sum() > self.k_per_group:
                # keep top-k by score
                top_idx = np.argsort(-scores_local)[:self.k_per_group]
                sel_mask[idx] = False
                sel_mask[idx][top_idx] = True

        self.selected_mask_ = sel_mask
        self.selected_features_ = list(pd.Series(self.columns_)[sel_mask])

        # build meta_
        meta["score"] = scores_all
        meta["selected"] = sel_mask
        self.meta_ = meta
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        return X.loc[:, self.selected_features_]

    def get_support(self, indices=False):
        if indices:
            return np.where(self.selected_mask_)[0]
        return self.selected_mask_


### utilities and logs

In [138]:
# ========== user-visible bundle ==========

class DataBundle:
    X: pd.DataFrame
    y: pd.Series
    meta: Dict[str, Any]

# ========== column utilities ==========

class StripDuplicateSuffix(BaseEstimator, TransformerMixin):
    """Drop duplicate columns that differ only by a trailing .<int> (keep first)."""
    def __init__(self, pattern=r"\.\d+$", verbose: bool = False):
        self.re = re.compile(pattern)
        self.verbose = verbose
        self.new_columns_ = None
        self.keep_idx_ = None
    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame): X = pd.DataFrame(X)
        orig_cols = list(X.columns)
        norm = [self.re.sub("", str(c)) for c in orig_cols]
        _, first_idx = np.unique(norm, return_index=True)
        self.keep_idx_ = np.sort(first_idx)
        self.new_columns_ = list(pd.Index(norm)[self.keep_idx_])
        return self
    def transform(self, X):
        if not isinstance(X, pd.DataFrame): X = pd.DataFrame(X)
        X = X.iloc[:, self.keep_idx_].copy()
        X.columns = self.new_columns_
        return X

class LogStep(BaseEstimator, TransformerMixin):
    """Wrap a transformer and print feature counts before/after."""
    def __init__(self, inner, name=None, desc="", verbose=True):
        self.inner = inner
        self.name = name or inner.__class__.__name__
        self.desc = desc
        self.verbose = verbose
    def get_params(self, deep=True):
        params = {"inner": self.inner, "name": self.name, "desc": self.desc, "verbose": self.verbose}
        if deep and hasattr(self.inner, "get_params"):
            for k, v in self.inner.get_params(deep=True).items():
                params[f"inner__{k}"] = v
        return params
    def set_params(self, **params):
        inner_params = {k.split("__", 1)[1]: v for k, v in params.items() if k.startswith("inner__")}
        for k, v in params.items():
            if not k.startswith("inner__"):
                setattr(self, k, v)
        if inner_params and hasattr(self.inner, "set_params"):
            self.inner.set_params(**inner_params)
        return self
    def __getattr__(self, name):
        return getattr(self.inner, name)
    def fit(self, X, y=None, **fit_params):
        self.inner.fit(X, y, **fit_params)
        return self
    def transform(self, X):
        before = getattr(X, "shape", (None, None))[1]
        Xt = self.inner.transform(X)
        after = getattr(Xt, "shape", (None, None))[1] if hasattr(Xt, "shape") else before
        if self.verbose:
            delta = None if (before is None or after is None) else (after - before)
            dtxt = "" if delta is None else f" ({'+' if delta>=0 else ''}{delta})"
            print(f"[prep/{self.name}] {self.desc} | {before} -> {after} features{dtxt}")
        return Xt
    def fit_transform(self, X, y=None, **fit_params):
        before = getattr(X, "shape", (None, None))[1]
        if hasattr(self.inner, "fit_transform"):
            Xt = self.inner.fit_transform(X, y, **fit_params)
        else:
            self.inner.fit(X, y, **fit_params)
            Xt = self.inner.transform(X)
        after = getattr(Xt, "shape", (None, None))[1] if hasattr(Xt, "shape") else before
        if self.verbose:
            delta = None if (before is None or after is None) else (after - before)
            dtxt = "" if delta is None else f" ({'+' if delta>=0 else ''}{delta})"
            print(f"[prep/{self.name}] {self.desc} | {before} -> {after} features{dtxt}")
        return Xt


### Data loader

In [139]:
# ========== data loader with feature flags ==========

class FeatureSetLoader:
    """
    Reads data via get_data(), chooses target, applies feature flags,
    and returns (X_train, y_train), (X_test, y_test).
    """
    def __init__(
        self,
        paths: Dict[str, str],
        dataset: str,
        target: str,
        *,
        use_get_data_fallback: bool = True,   # kept for compatibility (unused here)
        clahe: bool = False,
        drop_target_unknown_only: bool = True,  # kept for compatibility (handled in split_xy)
        numeric_only: bool = True,
        include_regex: Optional[str] = None,
        exclude_regex: Optional[str] = None,
        drop_columns: Optional[List[str]] = None,
        feature_flags: Optional[Dict[str, Any]] = None,
        verbose: bool = True,
        # NEW: do not read these from `paths` anymore
        test_size: Optional[float] = None,
        random_state: Optional[int] = None,
    ):
        self.paths = paths
        self.dataset = dataset

        # ---------- normalize target ----------
        tgt = target.strip().lower()
        if tgt == "idh":
            self.target = "IDH"
        elif tgt in ("cod", "1p19q", "label"):
            self.target = "COD"
        else:
            raise ValueError(f"Unsupported target: {target}")
        # --------------------------------------

        self.clahe = bool(clahe)
        self.drop_target_unknown_only = drop_target_unknown_only
        self.numeric_only = numeric_only
        self.include_regex = include_regex
        self.exclude_regex = exclude_regex
        self.drop_columns = set(drop_columns or [])

        ff = feature_flags or {}
        self.ff = {
            "drop_grade": bool(ff.get("drop_grade", False)),
            "include_prefixes": list(ff.get("include_prefixes", [])),
            "exclude_prefixes": list(ff.get("exclude_prefixes", [])),
            "keep": list(ff.get("keep", [])),
            "drop": list(ff.get("drop", [])),
        }

        self._verbose = verbose

        # >>> NEW: keep explicit control here (do not pull from paths) <<<
        self.test_size = test_size
        self.random_state = random_state

    @staticmethod
    def _drop_with_report(X: pd.DataFrame, cols: List[str], label: str, verbose: bool):
        present = [c for c in cols if c in X.columns]
        if present:
            before = X.shape[1]
            X = X.drop(columns=present)
            after = X.shape[1]
            if verbose:
                print(f"[loader/features] {label}: dropped {len(present)} -> {before} -> {after} features (-{before-after})")
        return X

    def load(self):
        from get_data import get_data

        # choose experiment string based on target
        if self.target == "IDH":
            experiment = f"{self.dataset}_IDH"
            label_col = "IDH"
        else:  # COD / 1p19q
            experiment = f"{self.dataset}_COD"
            label_col = "Label"   # after clean_df, 1p19q label is "Label"

        # Decide split params (prefer explicit ones; DO NOT read from paths)
        ts = self.test_size
        rs = self.random_state
        if self._verbose:
            print(f"[loader] get_data(experiment='{experiment}', test_size={ts}, random_state={rs}, clahe={self.clahe})")

        train_df, test_df = get_data(
            experiment=experiment,
            test_size=ts,
            random_state=rs,
            clahe=self.clahe,
        )

        # ---------------- Filtering logic ----------------
        def split_xy(df: pd.DataFrame):
            if self.target == "IDH":
                mask = df["IDH"] != -1
                y = df.loc[mask, "IDH"].astype(int)
            else:  # COD
                mask = df["Label"] != -1
                # optional cohort filtering (unchanged)
                if "IDH" in df.columns:
                    mask = mask & (df["IDH"] == 1)
                if "Grade" in df.columns:
                    mask = mask & (df["Grade"] < 4)
                y = df.loc[mask, "Label"].astype(int)

            X = df.loc[mask].drop(
                columns=[c for c in ["IDH", "1p19q", "Label", "Class"] if c in df.columns]
            )

            # user-specified drop_columns
            if self.drop_columns:
                before = X.shape[1]
                X = X.drop(columns=[c for c in self.drop_columns if c in X.columns], errors="ignore")
                after = X.shape[1]
                if self._verbose:
                    dropped = before - after
                    print(f"[loader] drop_columns={self.drop_columns} -> {before} -> {after} (-{dropped})")

            # numeric-only
            if self.numeric_only:
                prev = X.shape[1]
                X = X.select_dtypes(include=[np.number]).copy()
                if self._verbose:
                    print(f"[loader] numeric_only=True -> {prev} -> {X.shape[1]} features")

            # regex include/exclude (kept behavior if you want it here)
            if self.include_regex:
                prev = X.shape[1]
                X = X.filter(regex=self.include_regex)
                if self._verbose:
                    print(f"[loader] include_regex='{self.include_regex}' -> {prev} -> {X.shape[1]} features")
            if self.exclude_regex:
                prev = X.shape[1]
                X = X.drop(columns=X.filter(regex=self.exclude_regex).columns, errors="ignore")
                if self._verbose:
                    print(f"[loader] exclude_regex='{self.exclude_regex}' -> {prev} -> {X.shape[1]} features")

            return X, y
        # --------------------------------------------------

        X_train, y_train = split_xy(train_df)
        X_test,  y_test  = split_xy(test_df)

        if self._verbose:
            print(f"[loader] final: Xtrain{X_train.shape}, ytrain{y_train.shape}, "
                  f"Xtest{X_test.shape}, ytest{y_test.shape}")

        return (X_train, y_train), (X_test, y_test)


### Data split

In [140]:

# ========== splitters ==========

class KFoldProvider:
    def __init__(self, n_splits=10, random_state=42):
        self.kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    def iter(self, X, y):
        for tr_idx, te_idx in self.kf.split(X, y):
            yield tr_idx, te_idx

### Models

In [141]:
# ========== model wrappers ==========

class ModelWrapper:
    def fit(self, X, y): ...
    def predict_proba(self, X) -> np.ndarray: ...


class XGBWrapper(ModelWrapper):
    def __init__(
        self,
        scale_post_weight: bool = True,
        monitor_training: bool = False,
        eval_fraction: float = 0.2,
        early_stopping_rounds: int = 10000,
        print_every_n: int = 20,
        verbose: bool = True,
        random_state: int = 42,
        **kwargs
    ):
        from xgboost import XGBClassifier


        self.scale_post_weight = scale_post_weight
        self.monitor_training = monitor_training
        self.eval_fraction = eval_fraction
        self.early_stopping_rounds = early_stopping_rounds
        self.print_every_n = print_every_n
        self.verbose = verbose
        self.random_state = random_state

        # keep your kwargs, just set deterministic defaults if not provided
        if "random_state" not in kwargs:
            kwargs["random_state"] = random_state
        if "n_jobs" not in kwargs and "nthread" not in kwargs:
            kwargs["n_jobs"] = 1  # determinism (no logic change)
        if "tree_method" not in kwargs:
            kwargs["tree_method"] = "hist"  # same as you had
        if "eval_metric" not in kwargs:
            kwargs["eval_metric"] = ["auc", "logloss"]

        self.model = XGBClassifier(**kwargs)

    def fit(self, X, y):
        import numpy as np, pandas as pd, random as pyrandom, os

        # harden determinism without changing your algorithm
        pyrandom.seed(self.random_state)
        np.random.seed(self.random_state)
        os.environ.setdefault("PYTHONHASHSEED", "0")

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        pos, neg = int((y == 1).sum()), int((y == 0).sum())
        if self.scale_post_weight:
            spw = neg / max(pos, 1)
            self.model.set_params(scale_pos_weight=spw)
            if self.verbose:
                print(f"[model/xgb] fit: X{X.shape}, y(pos={pos},neg={neg}), spw={spw:.3f}")
        else:
            if self.verbose:
                print(f"[model/xgb] fit: X{X.shape}, y(pos={pos},neg={neg}), spw=OFF")

        if self.eval_fraction > 0:
            from sklearn.model_selection import StratifiedShuffleSplit
            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=self.eval_fraction, random_state=self.random_state
            )
            tr_idx, val_idx = next(sss.split(X, y))
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

            # keep your early stopping behavior; callbacks are optional
            try:
                import xgboost as xgb
                callbacks = []
                if self.print_every_n and hasattr(xgb, "callback") and hasattr(xgb.callback, "EvaluationMonitor"):
                    callbacks.append(xgb.callback.EvaluationMonitor(period=self.print_every_n, show_stdv=False))
                if self.early_stopping_rounds and hasattr(xgb, "callback") and hasattr(xgb.callback, "EarlyStopping"):
                    callbacks.append(
                        xgb.callback.EarlyStopping(
                            rounds=self.early_stopping_rounds, save_best=True, maximize=True,
                            data_name="validation_1", metric_name="auc"
                        )
                    )
                self.model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
            except Exception:
                # fallback keeps your original logic
                self.model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=True)
        else:
            self.model.fit(X, y)

        try:
            imp = np.asarray(self.model.feature_importances_, dtype=float)
            used, total = int((np.nan_to_num(imp) > 0).sum()), imp.size
            print(f"[model/xgb] features used (importance > 0): {used}/{total}")
        except Exception:
            pass
        return self

    def predict_proba(self, X):
        if self.verbose:
            print(f"[model/xgb] predict_proba: X{X.shape}")
        return self.model.predict_proba(X)[:, 1]


MODEL_REGISTRY = {
    "xgb": XGBWrapper
}


### Shap

In [142]:
# ========== SHAP + ranking ==========

class ShapExplainer:
    def __init__(self, cache=".cache/shap", kernel_background=200, kernel_sample=1000, verbose: bool = True):
        self.kernel_background = kernel_background
        self.kernel_sample = kernel_sample
        self._verbose = verbose
    def _cache_sig(self):
        return {"kb": self.kernel_background, "ks": self.kernel_sample}
    def explain(self, model_name: str, X_train: pd.DataFrame, model: ModelWrapper, cache_tag: str = None) -> pd.Series:
        import shap
        if not isinstance(X_train, pd.DataFrame):
            X_train = pd.DataFrame(X_train)
        if model_name == "xgb":
            if self._verbose:
                print(f"[shap] TreeExplainer on X{X_train.shape} (tag={cache_tag})")
            explainer = shap.TreeExplainer(model.model)
            sv = explainer.shap_values(X_train)
            if isinstance(sv, list):
                vals = np.abs(np.asarray(sv[1] if len(sv)==2 else np.mean(np.asarray(sv),0))).mean(axis=0)
            else:
                vals = np.abs(np.asarray(sv)).mean(axis=0)
        else:
            bg_n = min(self.kernel_background, len(X_train))
            samp_n = min(self.kernel_sample, len(X_train))
            if self._verbose:
                print(f"[shap] KernelExplainer bg={bg_n}, sample={samp_n} on X{X_train.shape} (tag={cache_tag})")
            bg = X_train.sample(bg_n, random_state=0)
            f = lambda z: model.predict_proba(pd.DataFrame(z, columns=X_train.columns))
            explainer = shap.KernelExplainer(f, bg)
            samp = X_train.sample(samp_n, random_state=1)
            sv = explainer.shap_values(samp)
            if isinstance(sv, list):
                vals = np.abs(np.asarray(sv[1] if len(sv)==2 else np.mean(np.asarray(sv),0))).mean(axis=0)
            else:
                vals = np.abs(np.asarray(sv)).mean(axis=0)
        series = pd.Series(vals, index=X_train.columns).sort_values(ascending=False)
        if self._verbose:
            print(f"[shap] top5: {series.head(5).index.tolist()}")
        return series

### BN & ANFIS

In [143]:
# ---------- BN branch ----------

class BNBuilder:
    def __init__(
        self, kind="tan", n_bins=5, disc_strategy="quantile", max_indegree=3,
        root: Optional[str] = None, root_strategy: str = "max_mi",
        verbose: bool = True,
    ):
        self.kind = kind; self.n_bins = int(n_bins); self.disc_strategy = disc_strategy
        self.max_indegree = max_indegree; self.root = root; self.root_strategy = root_strategy
        self._verbose = verbose
        try:
            from pgmpy.models import BayesianNetwork as _BN
        except Exception:
            from pgmpy.models import BayesianModel as _BN
        self._BN_cls = _BN
    def _cache_sig(self):
        return {"k": self.kind, "b": self.n_bins, "ds": self.disc_strategy, "mi": self.max_indegree}
    def _learn_tan_dag(self, Dd: pd.DataFrame):
        from sklearn.metrics import mutual_info_score
        from pgmpy.estimators import TreeSearch
        class_node = "Label"
        features = [c for c in Dd.columns if c != class_node]
        if not features:
            raise ValueError("[bn] TAN needs at least one feature besides the class")
        root = self.root
        if root is None or root not in features:
            if self.root_strategy == "first":
                root = features[0]
            else:
                mi_to_y = {f: mutual_info_score(Dd[f].astype(int), Dd[class_node].astype(int)) for f in features}
                root = max(mi_to_y, key=mi_to_y.get)
                if self._verbose:
                    top = sorted(mi_to_y.items(), key=lambda kv: kv[1], reverse=True)[:5]
                    print(f"[bn] TAN: root='{root}' (top MI to Label: {top})")
        else:
            if self._verbose: print(f"[bn] TAN: using user-specified root='{root}'")
        ts = TreeSearch(Dd, root_node=root)
        dag = ts.estimate(estimator_type="tan", class_node=class_node, show_progress=self._verbose)
        if self._verbose: print("[bn] TreeSearch TAN structure learned.")
        return dag
    def _learn_hc_dag(self, Dd: pd.DataFrame):
        from pgmpy.estimators import HillClimbSearch, BicScore
        hc = HillClimbSearch(Dd, scoring_method=BicScore(Dd))
        dag = hc.estimate(max_indegree=self.max_indegree)
        if "Label" not in dag.nodes(): dag.add_node("Label")
        if self._verbose: print(f"[bn] HillClimb structure learned (nodes={len(dag.nodes())}).")
        return dag
        
    def fit(self, X_train_topk: pd.DataFrame, y_train: pd.Series, cache_tag: str = None):
        from sklearn.preprocessing import KBinsDiscretizer
        from sklearn.impute import SimpleImputer
        from pgmpy.estimators import BayesianEstimator
        from pgmpy.inference import VariableElimination
        if self._verbose:
            print(f"[bn] fit: kind={self.kind}, n_bins={self.n_bins}, disc_strategy={self.disc_strategy}, "
                  f"indegree={self.max_indegree}, X{X_train_topk.shape}, tag={cache_tag}")

        # median impute to remove NaNs before discretization
        imputer = SimpleImputer(strategy="median")
        Xi = pd.DataFrame(imputer.fit_transform(X_train_topk),
                          columns=X_train_topk.columns, index=X_train_topk.index)

        df = Xi.copy(); df["Label"] = y_train.astype(int).values
        disc = KBinsDiscretizer(n_bins=self.n_bins, encode="ordinal", strategy=self.disc_strategy)
        Xd = pd.DataFrame(disc.fit_transform(df.drop(columns=["Label"])),
                          columns=Xi.columns, index=df.index).astype(int)
        Dd = pd.concat([Xd, df["Label"]], axis=1)

        dag = self._learn_tan_dag(Dd) if self.kind.lower()=="tan" else self._learn_hc_dag(Dd)
        model = self._BN_cls(list(dag.edges()))
        model.fit(Dd, estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=10)
        infer = VariableElimination(model)
        if self._verbose:
            print(f"[bn] model ready: nodes={len(model.nodes())}, edges={len(model.edges())}")
        return {"model": model, "disc": disc, "inference": infer, "imputer": imputer}

    def predict_proba(self, bn_pack, X_test_topk: pd.DataFrame) -> np.ndarray:
        imputer = bn_pack["imputer"]; disc = bn_pack["disc"]; model = bn_pack["model"]; infer = bn_pack["inference"]
        Xi = pd.DataFrame(imputer.transform(X_test_topk),
                          columns=X_test_topk.columns, index=X_test_topk.index)
        Xd = pd.DataFrame(disc.transform(Xi), columns=Xi.columns, index=Xi.index).astype(int)
        if self._verbose: print(f"[bn] predict: X{Xd.shape}")
        probs = []
        for _, row in Xd.iterrows():
            ev = {c: int(row[c]) for c in Xd.columns}
            q = infer.query(variables=["Label"], evidence=ev, show_progress=False)
            probs.append(float(q.values[1]))
        return np.array(probs)
    def predict_proba(self, bn_pack, X_test_topk: pd.DataFrame) -> np.ndarray:
        disc = bn_pack["disc"]; model = bn_pack["model"]; infer = bn_pack["inference"]
        Xd = pd.DataFrame(disc.transform(X_test_topk), columns=X_test_topk.columns, index=X_test_topk.index).astype(int)
        if self._verbose: print(f"[bn] predict: X{Xd.shape}")
        probs = []
        for _, row in Xd.iterrows():
            ev = {c: int(row[c]) for c in Xd.columns}
            q = infer.query(variables=["Label"], evidence=ev, show_progress=False)
            probs.append(float(q.values[1]))  # binary {0,1}
        return np.array(probs)


# ---------- ANFIS placeholder ----------

class ANFISBuilder:
    def __init__(self, variant="gmm", rules=8, epochs=50, verbose: bool = True):
        self.variant = variant; self.rules = rules; self.epochs = epochs; self._verbose = verbose


    def fit(self, X_train_topk: pd.DataFrame, y_train: pd.Series, cache_tag: str = None):
        from sklearn.linear_model import LogisticRegression
        from sklearn.impute import SimpleImputer
        if self._verbose:
            print(f"[anfis] (stub) training LogisticRegression: X{X_train_topk.shape} tag={cache_tag}")
        imputer = SimpleImputer(strategy="median")
        Xi = imputer.fit_transform(X_train_topk)
        model = LogisticRegression(max_iter=1000).fit(Xi, y_train)
        return {"model": model, "imputer": imputer}

    def predict_proba(self, anfis_model, X_test_topk: pd.DataFrame) -> np.ndarray:
        if self._verbose: print(f"[anfis] predict: X{X_test_topk.shape}")
        Xi = anfis_model["imputer"].transform(X_test_topk)
        return anfis_model["model"].predict_proba(Xi)[:, 1]

### Metrics

In [144]:
# ========== metrics ==========

def threshold_metrics(y_true, y_proba, thr=0.5) -> Dict[str, float]:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(y_proba, dtype=float).ravel()
    p = np.clip(p, 1e-7, 1 - 1e-7)
    y_pred = (p >= thr).astype(int)
    tn = np.sum((y_true==0)&(y_pred==0)); fp = np.sum((y_true==0)&(y_pred==1))
    fn = np.sum((y_true==1)&(y_pred==0)); tp = np.sum((y_true==1)&(y_pred==1))
    spec = tn / (tn + fp + 1e-9)
    bal_acc = 0.5 * (spec + (tp/(tp+fn+1e-9)))
    try: ll = log_loss(y_true, p)
    except ValueError: ll = log_loss(y_true, np.column_stack([1 - p, p]))
    return dict(
        auc=roc_auc_score(y_true, p),
        acc=accuracy_score(y_true, y_pred),
        prec=precision_score(y_true, y_pred, zero_division=0),
        rec=recall_score(y_true, y_pred),
        f1=f1_score(y_true, y_pred),
        spec=spec,
        bal_acc=bal_acc,
        logloss=ll,
    )


### Orchestrator

In [151]:
# ========== orchestrator (FULL: selector + XGB + optional SHAP→ANFIS with CV) ==========

class RadiomicsWorkflow:
    def __init__(self, config: Dict[str, Any]):
        self.cfg = config
        self.verbose = config.get("verbose", False)

        # loader
        loader_flags = config.get("loader", {})
        self.loader = FeatureSetLoader(
            paths=config["feature_paths"],
            dataset=config["dataset"],
            target=config["target"],
            clahe=loader_flags.get("clahe", False),
            drop_target_unknown_only=loader_flags.get("drop_target_unknown_only", True),
            numeric_only=loader_flags.get("numeric_only", True),
            include_regex=loader_flags.get("include_regex"),
            exclude_regex=loader_flags.get("exclude_regex"),
            drop_columns=loader_flags.get("drop_columns", []),
            feature_flags=config.get("features", {}),
            verbose=loader_flags.get("verbose", self.verbose),
            test_size=config.get("test_size"),
            random_state=config.get("seed"),
        )

        # model
        mdl_name = config["model"]["name"]
        self.model_name = mdl_name
        mdl_params = config["model"].get("params", {})
        mdl_verbose = config["model"].get("verbose", self.verbose)
        self.model = MODEL_REGISTRY[mdl_name](verbose=mdl_verbose, **mdl_params)

        # SHAP
        shap_flags = config.get("shap", {})
        self.shap = ShapExplainer(
            kernel_background=shap_flags.get("kernel_background", 200),
            kernel_sample=shap_flags.get("kernel_sample", 1000),
            verbose=shap_flags.get("verbose", self.verbose),
        )

        # BN
        bn_flags = config.get("bn", {"kind": "tan"})
        self.bn = BNBuilder(verbose=bn_flags.get("verbose", self.verbose), **bn_flags)

        # ---------------- ANFIS (IMPORTANT: handle enable here, not in builder) ----------------
        anfis_flags = (config.get("anfis", {}) or {}).copy()

        # workflow-level enable switch (DO NOT pass into ANFISBuilder)
        self.anfis_enabled = bool(
            anfis_flags.pop("enable", ("anfis" in config and config.get("anfis") is not None))
        )

        # workflow-level ANFIS feature selection controls
        self.anfis_top_k_shap = int(anfis_flags.pop("top_k_shap", 10))
        self.anfis_intersect_with_shap = bool(anfis_flags.pop("intersect_with_shap", True))

        # builder verbose is allowed, but do not keep in dict
        anfis_verbose = anfis_flags.pop("verbose", self.verbose)

        # only pass what ANFISBuilder supports
        allowed = {"variant", "rules", "epochs"}
        anfis_builder_kwargs = {k: v for k, v in anfis_flags.items() if k in allowed}

        self.anfis = ANFISBuilder(verbose=anfis_verbose, **anfis_builder_kwargs)

        # ---------------- preprocessing pipeline ----------------
        from sklearn.feature_selection import VarianceThreshold

        prep_flags = config.get("preprocess", {})
        steps = []

        selector = prep_flags.get("selector", "none").lower()
        do_scale = prep_flags.get("scale", True)
        with_mean = prep_flags.get("scale_with_mean", False)

        # 1) dedup
        if prep_flags.get("dedup", True):
            steps.append(("dedup", LogStep(
                StripDuplicateSuffix(verbose=prep_flags.get("dedup_verbose", self.verbose)),
                name="dedup", desc="remove duplicated cols", verbose=self.verbose
            )))

        # 2) variance threshold (before selection)
        if isinstance(prep_flags.get("var_threshold", None), (int, float)):
            steps.append(("var", LogStep(
                VarianceThreshold(threshold=float(prep_flags["var_threshold"])),
                name="var", desc="VarianceThreshold", verbose=self.verbose
            )))

        # 3) scale BEFORE lasso only
        if do_scale and selector == "lasso":
            steps.append(("scale", LogStep(
                DFStandardScaler(with_mean=with_mean),
                name="scale", desc="StandardScaler (pre-lasso)", verbose=self.verbose
            )))

        # 4) selector
        if selector == "topk":
            topk_groupby = config.get("topk_groupby", prep_flags.get("groupby", "class"))
            k = int(self.cfg.get("topk", 10))
            steps.append(("topk", LogStep(
                UniqueThenTopK(
                    k_per_group=k,
                    groupby=topk_groupby,
                    score_func=f_classif,
                    always_keep=self.cfg.get("topk_always_keep", [])
                ),
                name="topk",
                desc=f"UniqueThenTopK (k={k} per {topk_groupby})",
                verbose=self.verbose
            )))

        elif selector == "lasso":
            lasso_cfg = self.cfg.get("lasso", {})
            lasso_groupby = lasso_cfg.get("groupby", "class")
            steps.append(("lasso", LogStep(
                GroupedCoeThrLassoSelector(
                    alpha_grid=lasso_cfg.get("alpha_grid", None),
                    random_state=lasso_cfg.get("random_state", self.cfg.get("seed", 42)),
                    test_size=lasso_cfg.get("test_size", 0.5),
                    max_iter=lasso_cfg.get("max_iter", 5000),
                    groupby=lasso_groupby,
                    k_per_group=lasso_cfg.get("k_per_group", None),
                ),
                name="lasso",
                desc=f"Grouped Coe-Thr-Lasso (groupby={lasso_groupby})",
                verbose=self.verbose
            )))

        elif selector == "none":
            pass
        else:
            raise ValueError(f"preprocess.selector must be one of: topk, lasso, none. Got: {selector}")

        # 5) scale after selection if not lasso
        if do_scale and selector != "lasso":
            steps.append(("scale", LogStep(
                DFStandardScaler(with_mean=with_mean),
                name="scale", desc="StandardScaler", verbose=self.verbose
            )))

        self.prep = Pipeline(steps)
        self.force_numeric_before_pipeline = prep_flags.get("force_numeric_before_pipeline", True)

        self._print_flags = config.get("print_flags", True)
        self.branches = config.get("branches", [])


    # ---------- helpers ----------
    def _prep_train_test(self, X_tr, X_te, y_tr):
        if self.force_numeric_before_pipeline:
            num_cols = X_tr.select_dtypes(include=[np.number]).columns
            X_tr, X_te = X_tr[num_cols].copy(), X_te[num_cols].copy()

        X_tr_p = self.prep.fit_transform(X_tr, y_tr)
        X_te_p = self.prep.transform(X_te)

        # recover feature names after selection
        if "topk" in self.prep.named_steps:
            cols = list(self.prep.named_steps["topk"].selected_features_)
        elif "lasso" in self.prep.named_steps:
            cols = list(self.prep.named_steps["lasso"].selected_features_)
        elif "var" in self.prep.named_steps:
            # var runs before selection (in this workflow)
            if "dedup" in self.prep.named_steps:
                cols = list(self.prep.named_steps["dedup"].new_columns_)
            else:
                cols = list(X_tr.columns)
            mask = self.prep.named_steps["var"].get_support()
            cols = [c for c, keep in zip(cols, mask) if keep]
        elif "dedup" in self.prep.named_steps:
            cols = list(self.prep.named_steps["dedup"].new_columns_)
        else:
            cols = list(X_tr.columns)

        X_tr_p = pd.DataFrame(X_tr_p, columns=cols, index=X_tr.index)
        X_te_p = pd.DataFrame(X_te_p, columns=cols, index=X_te.index)
        return X_tr_p, X_te_p

    def _eval_once(self, model_obj, X_tr, y_tr, X_te, y_te, fold_tag=None, label="XGB"):
        model_obj.fit(X_tr, y_tr)
        proba_te = model_obj.predict_proba(X_te)
        base = threshold_metrics(y_te.values, proba_te, thr=self.cfg.get("thr", 0.5))

        if self.verbose:
            tag = f"[cv#{fold_tag}]" if fold_tag else "[cv]"
            df = pd.DataFrame([base])
            print(f"\n{tag} --------------------------- {label} metrics ---------------------------")
            print(df.to_string(index=False))
            print(f"\n{tag} ------------------------------------------------------------------")

        return base

    def _select_anfis_features(self, X_tr_p: pd.DataFrame, shap_series: pd.Series):
        """Choose ANFIS features from SHAP and/or pipeline-selected features."""
        # SHAP top-k
        shap_top = list(shap_series.head(min(self.anfis_top_k_shap, len(shap_series))).index)

        if self.anfis_intersect_with_shap:
            feats = [c for c in X_tr_p.columns if c in set(shap_top)]
            if len(feats) == 0:
                # fallback: at least something
                feats = list(X_tr_p.columns)[:max(1, min(self.anfis_top_k_shap, X_tr_p.shape[1]))]
        else:
            feats = list(X_tr_p.columns)

        return feats

    # ---------- run ----------
    def run(self) -> Dict[str, Any]:
        if self.verbose:
            print("[run] start")

        (X_tr, y_tr), (X_te, y_te) = self.loader.load()

        kfold = KFoldProvider(
            n_splits=self.cfg.get("cv", 5),
            random_state=self.cfg.get("seed", 42),
        )

        cv_xgb = []
        cv_anfis = []

        for i, (tr_idx, val_idx) in enumerate(kfold.iter(X_tr, y_tr), 1):
            print(f"\n--------------------------- Cross Validation set: {i} ---------------------------")
            X_tr_i, X_val_i = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
            y_tr_i, y_val_i = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]

            X_tr_p, X_val_p = self._prep_train_test(X_tr_i, X_val_i, y_tr_i)

            # ---- XGB ----
            xgb_metrics = self._eval_once(self.model, X_tr_p, y_tr_i, X_val_p, y_val_i, fold_tag=str(i), label="XGB")
            cv_xgb.append(xgb_metrics)

            # ---- ANFIS (optional) ----
            if self.anfis_enabled:
                # SHAP on TRAIN fold (after preprocessing)
                shap_series = self.shap.explain(self.model_name, X_tr_p, self.model, cache_tag=f"cv{i}")
                anfis_feats = self._select_anfis_features(X_tr_p, shap_series)

                # fit ANFIS model on train fold, eval on val fold
                anfis_model = self.anfis.fit(X_tr_p.loc[:, anfis_feats], y_tr_i.values)
                anfis_proba = self.anfis.predict_proba(anfis_model, X_val_p.loc[:, anfis_feats])
                anfis_metrics = threshold_metrics(y_val_i.values, anfis_proba, thr=self.cfg.get("thr", 0.5))

                if self.verbose:
                    tag = f"[cv#{i}]"
                    df = pd.DataFrame([anfis_metrics])
                    print(f"\n{tag} --------------------------- ANFIS metrics ---------------------------")
                    print(df.to_string(index=False))
                    print(f"\n{tag} ------------------------------------------------------------------")

                cv_anfis.append(anfis_metrics)

        # ---- final train on full train, test on held-out test ----
        X_tr_p, X_te_p = self._prep_train_test(X_tr, X_te, y_tr)

        self.model.fit(X_tr_p, y_tr)
        proba_te = self.model.predict_proba(X_te_p)
        test_metrics_xgb = threshold_metrics(y_te.values, proba_te, thr=self.cfg.get("thr", 0.5))

        test_metrics_anfis = None
        if self.anfis_enabled:
            shap_series_full = self.shap.explain(self.model_name, X_tr_p, self.model, cache_tag="train_full")
            anfis_feats_full = self._select_anfis_features(X_tr_p, shap_series_full)

            anfis_model_full = self.anfis.fit(X_tr_p.loc[:, anfis_feats_full], y_tr.values)
            anfis_proba_te = self.anfis.predict_proba(anfis_model_full, X_te_p.loc[:, anfis_feats_full])
            test_metrics_anfis = threshold_metrics(y_te.values, anfis_proba_te, thr=self.cfg.get("thr", 0.5))

        # ---- summaries ----
        summary_xgb = None
        summary_anfis = None

        if self.verbose:
            # XGB summary
            df_xgb = pd.DataFrame(cv_xgb)
            summary_xgb = pd.concat(
                [df_xgb.mean().to_frame().T.rename(index={0: "mean"}),
                 df_xgb.std(ddof=1).to_frame().T.rename(index={0: "std"})],
                axis=0
            )
            print("\n==================== XGB CV Summary (Mean/Std) ====================")
            print(summary_xgb.to_string(float_format=lambda x: f"{x:.4f}"))
            print("============================================================\n")

            # ANFIS summary (if enabled)
            if self.anfis_enabled and len(cv_anfis) > 0:
                df_a = pd.DataFrame(cv_anfis)
                summary_anfis = pd.concat(
                    [df_a.mean().to_frame().T.rename(index={0: "mean"}),
                     df_a.std(ddof=1).to_frame().T.rename(index={0: "std"})],
                    axis=0
                )
                print("\n==================== ANFIS CV Summary (Mean/Std) ====================")
                print(summary_anfis.to_string(float_format=lambda x: f"{x:.4f}"))
                print("============================================================\n")

                print("[test] XGB  :", test_metrics_xgb)
                print("[test] ANFIS:", test_metrics_anfis)

        return {
            "xgb_cv_results": cv_xgb,
            "xgb_cv_summary": summary_xgb,
            "xgb_test_metrics": test_metrics_xgb,

            "anfis_enabled": self.anfis_enabled,
            "anfis_cv_results": cv_anfis if self.anfis_enabled else None,
            "anfis_cv_summary": summary_anfis if self.anfis_enabled else None,
            "anfis_test_metrics": test_metrics_anfis if self.anfis_enabled else None,
        }


### Execution 

In [156]:
config = {
    "verbose": True,
    "print_flags": True,
    "split_verbose": True,

    "loader": {
        "drop_columns": ["Grade"],
        "verbose": True,
        "use_get_data_fallback": True,
        "drop_target_unknown_only": True
    },

    "preprocess": {
        "dedup": True,
        "selector": "topk",         # switch
        "scale": True,
        "scale_with_mean": False,
        # "var_threshold": 0.01,     # <- disable for now (can wipe too many)
        "dedup_verbose": True
    },

    "model": {
        "name": "xgb",
        "verbose": True,
        "params": {
            "n_estimators": 5000,
            "max_depth": 10,
            "tree_method": "hist",
            "learning_rate": 0.1,
            "subsample": 0.9,
            "objective": "binary:logistic",
            "gamma": 0.0,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
            "min_child_weight": 2,
            "colsample_bytree": 0.9,
            "random_state": 42,
            "n_jobs": -1,
            "max_bin": 512,
            "scale_post_weight": True,  # keep if your wrapper supports it
        }
    },

    "bn": {"kind": "tan", "n_bins": 5},

    # enable is optional if your workflow defaults to enabled when "anfis" exists,
    # but adding it makes it explicit and avoids ambiguity.
    "anfis": {
        "enable": True,
        "variant": "gmm",
        "rules": 7,
        "epochs": 75,
        "top_k_shap": 7,
        "intersect_with_shap": True
    },

    "shap": {"kernel_background": 200, "kernel_sample": 1000},

    "dataset": "EGD",
    "target": "1p19q",
    "feature_paths": {},

    "test_size": 0.15,
    "cv": 10,
    "seed": 42,
    "thr": 0.5,

    # ---- REQUIRED / IMPORTANT for selector="topk" ----
    "topk": 20,
    "topk_groupby": "class",
    "topk_always_keep": ["Age", "Sex"],
}

wf = RadiomicsWorkflow(config)
results = wf.run()


[run] start
[loader] get_data(experiment='EGD_COD', test_size=0.15, random_state=42, clahe=False)
EGD: total 1p19q = 259 | 1p19q+ = 73 | pos rate = 0.282
EGD: total idh = 467 | idh+ = 155 | pos rate = 0.332
[loader] drop_columns={'Grade'} -> 20217 -> 20216 (-1)
[loader] numeric_only=True -> 20216 -> 20216 features
[loader] drop_columns={'Grade'} -> 20217 -> 20216 (-1)
[loader] numeric_only=True -> 20216 -> 20216 features
[loader] final: Xtrain(133, 20216), ytrain(133,), Xtest(16, 20216), ytest(16,)

--------------------------- Cross Validation set: 1 ---------------------------
[prep/dedup] remove duplicated cols | 20216 -> 20216 features (+0)
[topk] scored 20214 features (non-const=18530, const=1684)
[topk] collapse contrasts: bases=6738 | kept=6738 | dropped=13476
1296 -> 20  (firstorder)
1728 -> 20  (glcm)
1008 -> 20  (gldm)
1152 -> 20  (glrlm)
1152 -> 20  (glszm)
360 -> 20  (ngtdm)
42 -> 20  (shape)
[topk] always-kept (not grouped): ['Age', 'Sex']
[prep/topk] UniqueThenTopK (k=20 p

In [153]:
config = {
    "verbose": True,
    "print_flags": True,
    "split_verbose": True,

    "loader": {
        "drop_columns": ["Grade"],
        "verbose": True,
        "use_get_data_fallback": True,
        "drop_target_unknown_only": True
    },

    "preprocess": {
        "dedup": True,
        "selector": "topk",      # <-- THIS is the switch
        "scale": True,
        "scale_with_mean": False,
        "var_threshold": 0.01,
        "dedup_verbose": True
    },

    "model": {
        "name": "xgb",
        "verbose": True,
        "params": {
            "n_estimators": 5000,
            "max_depth": 10,
            "tree_method": "hist",
            "learning_rate": 0.1,
            "subsample": 0.9,
            "objective": "binary:logistic",
            "gamma": 0.0,
            "reg_lambda": 1.0,
            "reg_alpha": 0.0,
            "min_child_weight": 2,
            "colsample_bytree": 0.9,
            "random_state": 42,
            "n_jobs": -1,
            "max_bin": 512,
            "scale_post_weight": True
        }
    },

    "bn": {"kind": "tan", "n_bins": 5},
    "anfis": {"variant": "gmm", "rules": 9, "epochs": 75, "top_k_shap": 9},
    "shap": {"kernel_background": 200, "kernel_sample": 1000},

    "dataset": "EGD",
    "target": "idh",
    "feature_paths": {},

    "test_size": 0.15,
    "cv": 10,
    "seed": 42,
    "thr": 0.5
}


wf = RadiomicsWorkflow(config)
results = wf.run()


[run] start
[loader] get_data(experiment='EGD_IDH', test_size=0.15, random_state=42, clahe=False)
EGD: total 1p19q = 259 | 1p19q+ = 73 | pos rate = 0.282
EGD: total idh = 467 | idh+ = 155 | pos rate = 0.332
[loader] drop_columns={'Grade'} -> 20217 -> 20216 (-1)
[loader] numeric_only=True -> 20216 -> 20216 features
[loader] drop_columns={'Grade'} -> 20217 -> 20216 (-1)
[loader] numeric_only=True -> 20216 -> 20216 features
[loader] final: Xtrain(398, 20216), ytrain(398,), Xtest(69, 20216), ytest(69,)

--------------------------- Cross Validation set: 1 ---------------------------
[prep/dedup] remove duplicated cols | 20216 -> 20216 features (+0)
[prep/var] VarianceThreshold | 20216 -> 13669 features (-6547)
[topk] scored 13669 features (non-const=13669, const=0)
[topk] collapse contrasts: bases=13669 | kept=13669 | dropped=0
13669 -> 10  (UNK)
[prep/topk] UniqueThenTopK (k=10 per class) | 13669 -> 10 features (-13659)
[prep/scale] StandardScaler | 10 -> 10 features (+0)
[prep/dedup] remo

In [121]:
# ================== ADD: ANFIS CV (XGB -> SHAP -> ANFIS) ==================
# This cell adds the missing ANFIS CV utilities and extends RadiomicsWorkflow.run()
# NOTE: It does NOT change your XGB pipeline; it just adds ANFIS evaluation.

from dataclasses import dataclass
import numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, precision_score,
    recall_score, f1_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, log_loss
)
from xgboost import DMatrix

@dataclass
class FoldArtifacts:
    xgb_pipe: object
    xgb_booster: object
    selected_features: list
    shap_top_features: list
    scaler_mean: np.ndarray
    scaler_scale: np.ndarray
    centers: np.ndarray
    sigmas: np.ndarray | None
    cov_type: str | None
    covariances: np.ndarray | None
    mixture_weights: np.ndarray | None

class ANFISZeroOrder(nn.Module):
    """Diagonal Gaussian membership - zero-order consequents."""
    def __init__(self, centers, sigmas, consequents):
        super().__init__()
        self.centers = nn.Parameter(torch.tensor(centers, dtype=torch.float32))
        self.log_sigma = nn.Parameter(torch.log(torch.tensor(sigmas, dtype=torch.float32)))
        self.consequents = nn.Parameter(torch.tensor(consequents, dtype=torch.float32))
    def forward(self, x):
        sigma = torch.exp(self.log_sigma)
        diff = x[:, None, :] - self.centers[None, :, :]
        s = -0.5 * ((diff / (sigma + 1e-12)) ** 2).sum(-1)
        w = torch.softmax(s, dim=1)
        return w @ self.consequents

def _train_anfis_from_selected(
    Xtr_proc: pd.DataFrame, ytr: np.ndarray,
    Xva_proc: pd.DataFrame, yva: np.ndarray,
    sel_names: list,
    *,
    fuzzy_antecedents_method: str = "gaussianmix",  # 'gaussianmix' | 'kmeans'
    K_rules: int = 8,
    gmm_cov: str = "auto",
    epochs: int = 75,
    lr: float = 1e-3,
    wd: float = 1e-4,
    seed: int = 0
):
    """Fit scaler -> antecedents (KMeans/GMM) -> zero-order ANFIS; return metrics + artifacts."""
    ytr = np.asarray(ytr, int); yva = np.asarray(yva, int)
    C = int(ytr.max()) + 1

    # scale & slice
    scaler = StandardScaler().fit(Xtr_proc.loc[:, sel_names].to_numpy())
    Ztr = scaler.transform(Xtr_proc.loc[:, sel_names].to_numpy())
    Zva = scaler.transform(Xva_proc.loc[:, sel_names].to_numpy())

    # drop near-constant
    std_tr = Ztr.std(axis=0)
    keep = np.where(std_tr >= 1e-12)[0]
    if keep.size == 0:
        raise ValueError("All selected features are near-constant after scaling.")
    if keep.size < len(sel_names):
        sel_names = [sel_names[i] for i in keep]
        Ztr, Zva = Ztr[:, keep], Zva[:, keep]
    M = Ztr.shape[1]

    # antecedents
    if fuzzy_antecedents_method == "kmeans":
        n_unique = np.unique(Ztr, axis=0).shape[0]
        K = max(1, min(K_rules, n_unique))
        km = KMeans(n_clusters=K, n_init=20, random_state=seed).fit(Ztr)
        centers = km.cluster_centers_
        sigmas = np.vstack([
            np.maximum((Ztr[km.labels_ == k].std(axis=0) if (km.labels_ == k).any() else Ztr.std(axis=0)), 1e-3)
            for k in range(K)
        ])
    else:
        cov_type = ("full" if (gmm_cov == "auto" and M <= 12) else "diag" if (gmm_cov == "auto") else gmm_cov)
        gmm = GaussianMixture(
            n_components=K_rules, covariance_type=cov_type,
            n_init=5, reg_covar=1e-6, random_state=seed
        ).fit(Ztr)
        K = gmm.n_components
        centers = gmm.means_.copy()
        if cov_type == "diag":
            sigmas = np.sqrt(gmm.covariances_.copy())
        else:
            sigmas = np.sqrt(np.stack([np.diag(Ck) for Ck in gmm.covariances_], 0))

    # init consequents from soft memberships
    diff = Ztr[:, None, :] - centers[None, :, :]
    s = -0.5 * ((diff / (sigmas + 1e-12)) ** 2).sum(axis=2)
    Wtr = np.exp(s - s.max(axis=1, keepdims=True))
    Wtr = Wtr / np.maximum(Wtr.sum(axis=1, keepdims=True), 1e-12)

    B = np.zeros((K, C))
    for k in range(K):
        wk = Wtr[:, k]; den = wk.sum() + 1e-8
        for c in range(C):
            B[k, c] = (wk * (ytr == c)).sum() / den
    consequents = np.log(B + 1e-6)

    model = ANFISZeroOrder(np.array(centers), np.array(sigmas), np.array(consequents))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    crit = nn.CrossEntropyLoss()

    Xtr_t = torch.tensor(Ztr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.long)
    Xva_t = torch.tensor(Zva, dtype=torch.float32)

    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = crit(model(Xtr_t), ytr_t)
        loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(Xva_t), dim=1).cpu().numpy()

    if probs.shape[1] == 2:
        auc = roc_auc_score(yva, probs[:, 1])
        ap = average_precision_score(yva, probs[:, 1])
        yhat_thr = (probs[:, 1] >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yva, yhat_thr, labels=[0, 1]).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        brier = brier_score_loss(yva, np.clip(probs[:, 1], 1e-7, 1-1e-7))
        ll = log_loss(yva, np.clip(np.c_[1-probs[:, 1], probs[:, 1]], 1e-7, 1-1e-7))
        prec = precision_score(yva, yhat_thr, zero_division=0)
        rec = recall_score(yva, yhat_thr, zero_division=0)
        f1 = f1_score(yva, yhat_thr, zero_division=0)
    else:
        auc = roc_auc_score(yva, probs, multi_class="ovr")
        ap = np.nan
        yhat_thr = probs.argmax(1)
        spec = np.nan
        brier = np.nan
        ll = log_loss(yva, np.clip(probs, 1e-7, 1-1e-7))
        prec = rec = f1 = np.nan

    metrics = dict(
        AUC=float(auc),
        AP=float(ap) if not np.isnan(ap) else np.nan,
        ACC=float(accuracy_score(yva, yhat_thr)),
        PREC=float(prec) if not np.isnan(prec) else np.nan,
        RECALL=float(rec) if not np.isnan(rec) else np.nan,
        F1=float(f1) if not np.isnan(f1) else np.nan,
        SPEC=float(spec) if not np.isnan(spec) else np.nan,
        BAL_ACC=float(balanced_accuracy_score(yva, yhat_thr)),
        BRIER=float(brier) if not np.isnan(brier) else np.nan,
        LOGLOSS=float(ll)
    )
    artifacts = dict(
        scaler_mean=scaler.mean_, scaler_scale=scaler.scale_,
        centers=np.array(centers), sigmas=np.array(sigmas),
        cov_type=None, covariances=None, mixture_weights=None,
        sel_names=sel_names
    )
    return metrics, model, artifacts

def _shap_topk_from_booster(booster, Xtr_sel: pd.DataFrame, ytr: np.ndarray, top_k: int):
    dtr = DMatrix(Xtr_sel, label=ytr)
    contribs = booster.predict(dtr, pred_contribs=True, approx_contribs=False)
    phi = contribs[:, :-1]  # drop bias
    names = dtr.feature_names
    mean_abs = np.mean(np.abs(phi), axis=0)
    order = np.argsort(mean_abs)[::-1]
    return [names[i] for i in order[:min(top_k, len(names))]]

# ---- Extend RadiomicsWorkflow.run to print ANFIS CV ----
if "RadiomicsWorkflow" in globals():
    _old_run = RadiomicsWorkflow.run

    def _run_with_anfis(self):
        out = _old_run(self)

        anfis_cfg = self.cfg.get("anfis", {}) or {}
        if not anfis_cfg.get("cv_enable", True):
            return out

        (X_tr, y_tr), _ = self.loader.load()
        kfold = KFoldProvider(
            n_splits=self.cfg.get("cv", 5),
            random_state=self.cfg.get("seed", 42),
        )

        rows = []
        for i, (tr_idx, val_idx) in enumerate(kfold.iter(X_tr, y_tr), 1):
            X_tr_i, X_val_i = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
            y_tr_i = y_tr.iloc[tr_idx].values
            y_val_i = y_tr.iloc[val_idx].values

            X_tr_p, X_val_p = self._prep_train_test(X_tr_i, X_val_i, y_tr.iloc[tr_idx])
            self.model.fit(X_tr_p, y_tr.iloc[tr_idx])

            feat_cols = list(X_tr_p.columns)

            shap_k = int(anfis_cfg.get("top_k_shap", min(10, len(feat_cols))))
            shap_feats = feat_cols[:shap_k]
            try:
                clf = getattr(self.model, "model", None)
                booster = clf.get_booster() if clf is not None and hasattr(clf, "get_booster") else None
                if booster is not None:
                    booster.feature_names = feat_cols
                    shap_feats = _shap_topk_from_booster(booster, X_tr_p[feat_cols], y_tr_i, shap_k)
            except Exception:
                pass

            anfis_feats = feat_cols
            if anfis_cfg.get("intersect_with_shap", True):
                anfis_feats = sorted(set(feat_cols).intersection(set(shap_feats)))
                if len(anfis_feats) == 0:
                    anfis_feats = shap_feats
            elif anfis_cfg.get("use_shap_only", False):
                anfis_feats = shap_feats

            metrics_anfis, _, _ = _train_anfis_from_selected(
                X_tr_p, y_tr_i,
                X_val_p, y_val_i,
                anfis_feats,
                fuzzy_antecedents_method=("gaussianmix" if anfis_cfg.get("variant", "gmm") == "gmm" else "kmeans"),
                K_rules=int(anfis_cfg.get("rules", 8)),
                gmm_cov=str(anfis_cfg.get("gmm_cov", "auto")),
                epochs=int(anfis_cfg.get("epochs", 75)),
                lr=float(anfis_cfg.get("lr", 1e-3)),
                wd=float(anfis_cfg.get("wd", 1e-4)),
                seed=int(self.cfg.get("seed", 42)),
            )

            if self.verbose:
                print(f"\n[cv#{i}] --------------------------- ANFIS metrics ---------------------------")
                dfm = pd.DataFrame([{
                    "auc": metrics_anfis["AUC"],
                    "acc": metrics_anfis["ACC"],
                    "prec": metrics_anfis.get("PREC", np.nan),
                    "rec": metrics_anfis.get("RECALL", np.nan),
                    "f1": metrics_anfis.get("F1", np.nan),
                    "spec": metrics_anfis.get("SPEC", np.nan),
                    "bal_acc": metrics_anfis.get("BAL_ACC", np.nan),
                    "logloss": metrics_anfis.get("LOGLOSS", np.nan),
                }])
                print(dfm.to_string(index=False))
                print(f"\n[cv#{i}] ------------------------------------------------------------------")

            rows.append({
                "fold": i,
                **{f"ANFIS_{k.lower()}": v for k, v in metrics_anfis.items()},
                "n_feats_pipe": len(feat_cols),
                "n_feats_anfis": len(anfis_feats),
            })

        out["anfis_cv_results"] = pd.DataFrame(rows)

        if self.verbose:
            df = out["anfis_cv_results"]
            print("\n==================== ANFIS CV Summary (Mean/Std) ====================")
            show = {}
            for key in ["AUC", "ACC", "PREC", "RECALL", "F1", "SPEC", "BAL_ACC", "LOGLOSS"]:
                col = f"ANFIS_{key.lower()}"
                if col in df.columns:
                    show[key.lower()] = [df[col].mean(), df[col].std(ddof=1)]
            sm = pd.DataFrame(show, index=["mean", "std"]).T
            print(sm.to_string(float_format=lambda x: f"{x:.4f}"))
            print("====================================================================")

        return out

    RadiomicsWorkflow.run = _run_with_anfis
    print('[patch] RadiomicsWorkflow.run extended: ANFIS per-fold + summary. Output key: results["anfis_cv_results"].')


[patch] RadiomicsWorkflow.run extended: ANFIS per-fold + summary. Output key: results["anfis_cv_results"].
